# Project 13 — Train a Small LM (100M / 300M / 1B config comparison + throughput dashboard)

```
TinyShakespeare -> char ids -> 3 proxy GPTs (XS / S / M, 0.1M -> 1.2M params)
  -> brief CPU training each -> loss + tok/s per config
  -> verified param formula -> 100M / 300M / 1B memory + throughput projections
  -> throughput dashboard (CSVs + PNG)
```

Goal: feel how loss, speed, and memory scale with model size, then project what
100M / 300M / 1B training would cost — without ever training one here.

Learning objectives:
1. Count transformer params analytically and verify against torch (exact match).
2. Measure throughput (tok/s) per config and find the per-token cost of a param.
3. Project memory (weights + grads + Adam states) and tok/s for 100M / 300M / 1B.
4. Show batch-size scaling: when bigger batches stop helping on one CPU.


In [ ]:
import os # need os to create results/ and check cwd
import random # need random to seed stdlib RNG
import time # need time to measure tok/s and training time
import csv # need csv to write config/throughput tables
import math # need math for ln(V) baseline, perplexity, and scaling math
import urllib.request # need urllib to try downloading TinyShakespeare (offline fallback)
import numpy as np # need numpy for data splits and batch indexing
import torch # need torch for the GPT models and training loops
import torch.nn.functional as F # need F for cross-entropy loss
import matplotlib # need matplotlib for dashboard plots
matplotlib.use('Agg') # force headless PNG backend (no display needed)
import matplotlib.pyplot as plt # need pyplot for loss/throughput figures
SEED = 13 # fixed seed: same init, same batches, comparable configs
random.seed(SEED) # seed stdlib
np.random.seed(SEED) # seed numpy
torch.manual_seed(SEED) # seed torch
torch.set_num_threads(max(1, os.cpu_count() // 2)) # cap threads: stable timing on shared boxes
os.makedirs('./results', exist_ok=True) # create results/ for CSVs and PNGs
DEV = 'cuda' if torch.cuda.is_available() else 'cpu' # GPU if present else CPU
print(f"device={DEV} torch={torch.__version__}") # env line for the README

In [ ]:
# ---- data: TinyShakespeare download with offline synthetic fallback ----
URL = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt' # canonical tiny-shakespeare source
MAX_CHARS = 120000 # cap corpus: trains fast on CPU yet keeps Shakespeare flavor
def synthetic_text(n): # deterministic fallback when offline: pseudo-early-modern loop
    words = ['to', 'be', 'thou', 'thee', 'king', 'queen', 'night', 'day', 'love', 'death', 'heart', 'lord', 'sweet', 'fair', 'dark', 'light', 'shall', 'will', 'not', 'and'] # small vocab pool
    rng = random.Random(SEED) # seeded: identical fallback every run
    out = [] # collect words
    while sum(len(w) + 1 for w in out) < n: # fill until n chars
        out.append(rng.choice(words)) # pick next word
    return ' '.join(out)[:n] # join and trim to exactly n
try: # attempt download first (needs network)
    with urllib.request.urlopen(URL, timeout=15) as r: # 15s timeout: never hang headless runs
        text = r.read().decode('utf-8')[:MAX_CHARS] # decode and cap length
    SRC = 'tinyshakespeare' # record real-data provenance
except Exception as e: # offline or blocked: fall back
    print(f"download failed ({e}); using synthetic fallback") # say why
    text = synthetic_text(MAX_CHARS) # deterministic substitute
    SRC = 'synthetic-fallback' # record fallback provenance
chars = sorted(set(text)) # vocab = sorted unique chars (char-level tokenizer)
V = len(chars) # vocab size (~65 for Shakespeare)
stoi = {c: i for i, c in enumerate(chars)} # char -> id map
itos = {i: c for c, i in stoi.items()} # id -> char map
encode = lambda s: [stoi[c] for c in s] # tokenizer: string -> id list
decode = lambda ids: ''.join(itos[i] for i in ids) # detokenizer: id list -> string
data = torch.tensor(encode(text), dtype=torch.long) # full corpus as id tensor
ntr = int(0.9 * len(data)) # 90/10 train/val split point
tr, va = data[:ntr], data[ntr:] # train and val id tensors
print(f"src={SRC} chars={len(text)} vocab={V} train={len(tr)} val={len(va)}") # dataset card
print(f"uniform baseline loss=ln(V)={math.log(V):.4f} (any trained loss must beat this)") # baseline: random guessing
# ---- batching: random (x, y) pairs with y = x shifted by one ----
CTX, BATCH = 48, 24 # context length and batch size: 24*48=1152 tok/iter
def batch(split, n=BATCH): # sample n random windows from a split
    src = tr if split == 'train' else va # pick corpus
    ix = torch.randint(0, len(src) - CTX - 1, (n,)) # random start offsets
    x = torch.stack([src[i:i + CTX] for i in ix]) # inputs: CTX ids
    y = torch.stack([src[i + 1:i + CTX + 1] for i in ix]) # targets: same window shifted +1
    return x, y # (B, T) pair

In [ ]:
# ---- model factory: TinyGPT with configurable depth/width + analytic param math ----
class Block(torch.nn.Module): # one pre-norm transformer block (GPT-2 style)
    def __init__(self, d, h): # build block: width d, heads h
        super().__init__() # init nn.Module machinery
        self.ln1 = torch.nn.LayerNorm(d) # first norm: stabilizes attention inputs
        self.qkv = torch.nn.Linear(d, 3 * d, bias=False) # fused QKV: one matmul, split later
        self.proj = torch.nn.Linear(d, d, bias=False) # output proj: mixes heads back to d
        self.ln2 = torch.nn.LayerNorm(d) # second norm: stabilizes MLP inputs
        self.fc1 = torch.nn.Linear(d, 4 * d, bias=False) # MLP expand: d -> 4d features
        self.fc2 = torch.nn.Linear(4 * d, d, bias=False) # MLP contract: 4d -> d back
        self.h = h # store head count for the forward split
    def forward(self, x): # x: (B, T, d) token representations
        B, T, d = x.shape # unpack batch, time, width
        qkv = self.qkv(self.ln1(x)) # norm then fused QKV projection
        q, k, v = qkv.chunk(3, dim=-1) # split into Q, K, V views
        q = q.view(B, T, self.h, d // self.h).transpose(1, 2) # (B, h, T, dh) query heads
        k = k.view(B, T, self.h, d // self.h).transpose(1, 2) # (B, h, T, dh) key heads
        v = v.view(B, T, self.h, d // self.h).transpose(1, 2) # (B, h, T, dh) value heads
        w = (q @ k.transpose(-2, -1)) / math.sqrt(d // self.h) # scaled scores: similarity / sqrt(dh)
        mask = torch.ones(T, T, dtype=torch.bool).tril() # causal mask: position i sees <= i
        w = w.masked_fill(~mask, float('-inf')) # forbid future: -inf becomes 0 after softmax
        w = w.softmax(dim=-1) # attention weights: rows sum to 1
        a = (w @ v).transpose(1, 2).reshape(B, T, d) # weighted values merged back to (B, T, d)
        x = x + self.proj(a) # residual 1: gradient highway around attention
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x)))) # residual 2: norm -> expand -> gelu -> contract
        return x # (B, T, d) transformed representations
class TinyGPT(torch.nn.Module): # decoder-only char LM: embed -> blocks -> head
    def __init__(self, L, d, h, V, ctx): # build LM: L blocks, width d, h heads
        super().__init__() # init nn.Module machinery
        self.tok = torch.nn.Embedding(V, d) # token table: id -> d-dim vector
        self.pos = torch.nn.Embedding(ctx, d) # position table: order info (P11 showed it matters)
        self.blocks = torch.nn.ModuleList([Block(d, h) for _ in range(L)]) # L stacked blocks
        self.ln = torch.nn.LayerNorm(d) # final norm before the head
        self.head = torch.nn.Linear(d, V, bias=False) # LM head: d -> vocab logits (untied)
    def forward(self, x): # x: (B, T) token ids
        B, T = x.shape # unpack shapes
        p = torch.arange(T, device=x.device) # positions 0..T-1 on the right device
        e = self.tok(x) + self.pos(p) # embed: meaning + order
        for b in self.blocks: # apply each block in sequence
            e = b(e) # refine representations residually
        return self.head(self.ln(e)) # (B, T, V) next-token logits
def count_params(m): # count trainable params via torch (ground truth)
    return sum(p.numel() for p in m.parameters() if p.requires_grad) # sum sizes of grad-tracked tensors
def gpt_params(L, H, V, ctx): # analytic param budget (same accounting as P12)
    emb = V * H + ctx * H # token table + position table
    per_block = 4 * H * H + 8 * H * H + 4 * H # attn (4H^2) + MLP (8H^2) + 2 LayerNorms (weight+bias each)
    out = 2 * H + V * H # final LN (weight+bias) + untied LM head
    return emb + L * per_block + out # total scalar: must match torch count
PROXIES = [('XS', 2, 64, 4), ('S', 4, 96, 4), ('M', 6, 128, 4)] # (name, layers, width, heads): CPU-trainable
ITERS = {'XS': 150, 'S': 120, 'M': 80} # iters per proxy: bigger models get fewer (time budget)
for name, L, d, h in PROXIES: # verify analytic math against torch for every proxy
    m = TinyGPT(L, d, h, V, CTX) # build the exact model we will train
    a, t = gpt_params(L, d, V, CTX), count_params(m) # analytic vs torch ground truth
    assert a == t, f"{name}: analytic {a} != torch {t}" # accounting must be exact, else projections lie
    print(f"{name}: L={L} d={d} h={h} params={t:,}") # config card with verified count
# ---- reference configs: analytic only (too big to train here, BPE-scale vocab) ----
REFS = [('100M-class', 12, 768, 12, 32000, 2048), ('300M-class', 24, 1024, 16, 32000, 2048), ('1B-class', 24, 2048, 16, 32000, 2048)] # (name, L, H, heads, V, ctx): GPT-style shapes
print(f"{'config':<12}{'params':>12}{'infer-bf16':>12}{'train-bf16':>12}") # dashboard header
for name, L, H, nh, Vv, ctx in REFS: # project memory for each reference scale
    P = gpt_params(L, H, Vv, ctx) # analytic param total (heads do not change the count)
    infer_gb = 2 * P / 1e9 # inference: bf16 weights only, 2 bytes/param
    train_gb = 14 * P / 1e9 # training: bf16 weights(2) + fp32 grad(4) + Adam m,v(8) bytes/param
    print(f"{name:<12}{P / 1e6:>11.1f}M{infer_gb:>11.2f}G{train_gb:>11.2f}G") # memory projection row

In [ ]:
# ---- train each proxy briefly; record loss, time, tok/s ----
def val_loss(m, n=20): # mean CE over n val batches (no grad)
    m.eval() # eval mode (no dropout here; marks intent)
    tot, cnt = 0.0, 0 # accumulate loss x tokens and token count
    with torch.no_grad(): # no graph: less memory, faster
        for _ in range(n): # n random val windows
            xb, yb = batch('val') # sample a val batch
            xb, yb = xb.to(DEV), yb.to(DEV) # move batch to device
            l = F.cross_entropy(m(xb).view(-1, V), yb.view(-1)) # batch CE over all positions
            tot += l.item() * xb.numel() # weight batch loss by its tokens
            cnt += xb.numel() # count tokens
    m.train() # back to train mode for the caller
    return tot / cnt # mean loss per token
def train_cfg(name, L, d, h, iters, lr=3e-4): # train one proxy config from scratch
    m = TinyGPT(L, d, h, V, CTX).to(DEV) # fresh model on device
    opt = torch.optim.AdamW(m.parameters(), lr=lr) # AdamW: decoupled weight decay default
    l0 = val_loss(m) # loss before training (should sit near ln(V))
    t0 = time.time() # start wall clock
    for it in range(iters): # fixed iteration budget for this config
        xb, yb = batch('train') # random (B, T) window, y = x shifted +1
        xb, yb = xb.to(DEV), yb.to(DEV) # move batch to device
        loss = F.cross_entropy(m(xb).view(-1, V), yb.view(-1)) # forward + CE over positions
        opt.zero_grad() # clear old grads (else they accumulate)
        loss.backward() # backprop: dLoss/dW for every param
        opt.step() # AdamW update step
    dt = time.time() - t0 # wall seconds for the whole run
    toks = iters * BATCH * CTX # total training tokens seen
    row = {'name': name, 'L': L, 'd': d, 'h': h, 'params': count_params(m), # config identity
           'iters': iters, 'tokens': toks, 'seconds': round(dt, 2), 'tok_s': round(toks / dt, 1), # throughput facts
           'init_loss': round(l0, 4), 'train_loss': round(loss.item(), 4), # before/after training loss
           'val_loss': round(val_loss(m), 4)} # held-out loss (the number that matters)
    row['val_ppl'] = round(math.exp(row['val_loss']), 2) # perplexity: exp(CE), lower is better
    print(row) # one-line card per config
    return row # hand the row back for CSV + plots
rows = [train_cfg(name, L, d, h, ITERS[name]) for name, L, d, h in PROXIES] # train XS, S, M in order
with open('./results/configs.csv', 'w', newline='') as f: # save the comparison table
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())) # header = row keys
    w.writeheader() # write header once
    w.writerows(rows) # write one row per proxy config
print('saved ./results/configs.csv') # confirm artifact for the README

In [ ]:
# ---- throughput dashboard: scaling plots + 100M/300M/1B projections + batch sweep ----
anchor = rows[1] # S config anchors projections (middle size, least noisy timing)
rate = anchor['tok_s'] * anchor['params'] # fitted rate: tok/s falls as 1/params (linear-dominated regime)
proj = [] # projection rows for the dashboard
for name, L, H, nh, Vv, ctx in REFS: # same reference shapes as the memory table
    P = gpt_params(L, H, Vv, ctx) # analytic params again
    proj.append({'config': name, 'params_M': round(P / 1e6, 1), # scale label
                 'infer_bf16_GB': round(2 * P / 1e9, 2), 'train_bf16_GB': round(14 * P / 1e9, 2), # memory columns
                 'proj_tok_s_cpu': round(rate / P, 3)}) # projected tok/s on THIS cpu via 1/P law
with open('./results/projections.csv', 'w', newline='') as f: # save projections
    w = csv.DictWriter(f, fieldnames=list(proj[0].keys())) # header from keys
    w.writeheader() # write header
    w.writerows(proj) # one row per reference scale
print('saved ./results/projections.csv') # confirm artifact
# ---- what-if: batch-size sweep on a fresh S model (forward+backward timing) ----
ms = TinyGPT(4, 96, 4, V, CTX).to(DEV) # fresh S model: timing only, no training needed
opts = torch.optim.AdamW(ms.parameters(), lr=3e-4) # optimizer so backward is timed too
sweep = [] # (batch, tok/s) rows
for B in [8, 16, 32, 64]: # batch sizes to probe for saturation
    xb = torch.randint(0, V, (B, CTX)).to(DEV) # random ids: timing does not need real text
    yb = torch.randint(0, V, (B, CTX)).to(DEV) # random targets likewise
    t0 = time.time() # start sweep timer
    for _ in range(12): # a few iters per batch size (mean timing is fine here)
        loss = F.cross_entropy(ms(xb).view(-1, V), yb.view(-1)) # forward + CE
        opts.zero_grad() # clear grads
        loss.backward() # backward pass
        opts.step() # optimizer step
    dt = time.time() - t0 # seconds for 12 iters
    sweep.append({'batch': B, 'tok_s': round(12 * B * CTX / dt, 1)}) # throughput for this batch
    print(sweep[-1]) # one line per batch size
with open('./results/sweep.csv', 'w', newline='') as f: # save the sweep
    w = csv.DictWriter(f, fieldnames=['batch', 'tok_s']) # two-column table
    w.writeheader() # header
    w.writerows(sweep) # rows
print('saved ./results/sweep.csv') # confirm artifact
# ---- dashboard figure: loss vs params, tok/s vs params (+projections), tok/s vs batch ----
fig, ax = plt.subplots(1, 3, figsize=(12, 3.5)) # three panels side by side
ps = [r['params'] for r in rows] # proxy param counts for x-axes
ax[0].plot(ps, [r['val_loss'] for r in rows], 'o-') # val loss vs params
ax[0].set_xscale('log') # log-x: sizes span an order of magnitude
ax[0].set_xlabel('params (log)') # label x
ax[0].set_ylabel('val CE') # label y
ax[0].set_title('bigger -> lower loss') # takeaway title
ax[0].axhline(math.log(V), ls='--', c='gray') # dashed: untrained ln(V) floor
for r in rows: # annotate each point with its name
    ax[0].annotate(r['name'], (r['params'], r['val_loss'])) # text label at the point
ax[1].loglog(ps, [r['tok_s'] for r in rows], 'o-', label='measured') # measured tok/s, log-log
ax[1].loglog([gpt_params(L, H, Vv, ctx) for _, L, H, _, Vv, ctx in REFS], [p['proj_tok_s_cpu'] for p in proj], 'x--', label='projected') # 1/P projections
ax[1].set_xlabel('params (log)') # label x
ax[1].set_ylabel('tok/s (log)') # label y
ax[1].set_title('tok/s falls ~ 1/params') # takeaway title
ax[1].legend(fontsize=8) # distinguish measured vs projected
ax[2].plot([s['batch'] for s in sweep], [s['tok_s'] for s in sweep], 'o-') # sweep curve
ax[2].set_xlabel('batch size') # label x
ax[2].set_ylabel('tok/s') # label y
ax[2].set_title('batch scaling saturates') # takeaway title
fig.suptitle(f"P13 throughput dashboard ({DEV}, src={SRC})") # overall title with provenance
fig.tight_layout() # avoid overlapping labels
fig.savefig('./results/dashboard.png', dpi=100) # save the dashboard figure
print('saved ./results/dashboard.png') # confirm artifact
print(f"anchor S: {anchor['tok_s']} tok/s at {anchor['params']:,} params -> rate={rate:.3g} param-tok/s") # fitted scaling law
print(f"1B-class projection: {proj[2]['proj_tok_s_cpu']} tok/s on this CPU, {proj[2]['train_bf16_GB']} GB to train in bf16") # headline: why 1B needs a GPU

Cleanup / next steps:
- Artifacts: `results/configs.csv` (measured), `results/projections.csv` (100M/300M/1B),
  `results/sweep.csv` (batch sweep), `results/dashboard.png`.
- Rerun on the DGX Spark: same notebook, tok/s rows jump ~50–100x, projections re-anchor.
- Next (P14): take the S proxy, teach it a new skill cheaply — SFT vs LoRA vs QLoRA vs full FT.
